In [3]:
import numpy as np
import pandas as pd
import itertools
import matplotlib.pyplot as plt
from matplotlib import cm
from datetime import date
import re
from tqdm import tqdm
import time
# from pybedtools import BedTool
import random
import math
import pickle
import seaborn as sns
from scipy.stats import percentileofscore, pearsonr, rankdata, fisher_exact, describe, hypergeom
from scipy.spatial.distance import jaccard, euclidean, cosine, pdist, squareform
import scipy as sp
import functools
import operator
import gzip
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from scipy.cluster import hierarchy
import sys
import os
from functools import partial
from contextlib import closing
from multiprocessing import Pool
import re
import h5py
from collections import Counter
# from region_comparison_combine_scores_CT20220810 import combine_scores
from operator import is_not

today = date.today()

In [2]:
def kde_wrapper(data, bw=0.1):
    fig = plt.figure()
    fig.patch.set_facecolor('white')
    
    sns.kdeplot(data, bw_adjust=bw)
    print(describe(data))
    return fig

**z score data too large**

In [4]:
mapped_counts_filename = '/home/ctrader/region_similarity/data/rank_based/regions_25kb_CT20221213.TFsignalmapped.bed'

In [160]:
# load count data
counts = []
means = []

f = open(mapped_counts_filename, 'r')

# array to track scores for each factor
factors = f.readline().strip().split('\t')[3:]
factor_scores = []
for i in range(len(factors)):
    factor_scores.append([])

# for each region
for line in f.readlines():  # not header
    line = re.split('[\t|]', line.strip())
    # number of each motif for region
    counts.append(line[3::3])
    # average score for each motif for region
    means.append(line[4::3])

    # inidvidual scores for each region
    line_scores = line[5::3]
    line_scores = np.char.rsplit(line_scores, ';')
    for factor in range(len(factors)):
        # test to see if first element is ''
        if line_scores[factor][0]:
            factor_scores[factor].extend(list(np.float_(line_scores[factor])))
f.close()

In [169]:
counts = np.int_(counts)
means = np.array(means)
means = np.where(means == 'NAN', np.nan, means).astype('float32')

In [172]:
# find mean and std for each factor
factor_means = np.array([])
factor_std = np.array([])
for factor in range(len(factors)):
    factor_means = np.append(factor_means, np.mean(factor_scores[factor]))
    factor_std = np.append(factor_std, np.std(factor_scores[factor]))

# z score
z = ((means - factor_means) / factor_std * np.sqrt(counts))
# one sided
z = np.where(((z < 0) | (np.isnan(z))), 0, z)

**Problem with finding the top 1000 values** 

In [2]:
# calculates sigmoid value of rank
def sigmoid_func(ranked_arr, scaling_factor=5):

    # max rank
    max_rank = np.max(ranked_arr, axis=0)

    # scaling between [-scaling_factor, scaling_factor]
    x = (ranked_arr / (max_rank / (scaling_factor * 2))) - scaling_factor
    sig_arr = 1 / (1 + np.exp(-x))
    return sig_arr


# top 1000 regions and values
def find_top_regions(sim_m):
    top_regions_all = []
    top_vals_all = []
    for i in range(len(sim_m)):
        top_regions = np.argsort(-sim_m[i])[:1001]
        top_regions = top_regions[top_regions != i][:1000]
        top_regions_all.append(top_regions)
        top_vals_all.append(sim_m[i, top_regions])
    top_regions_all = np.array(top_regions_all)
    top_vals_all = np.array(top_vals_all)
    return top_regions_all, top_vals_all


def rank_regions(arr):
    # rank each region
    ranked_regions = rankdata(arr, axis=0)

    # find the number of regions with 0 values for each component
    comp_num_zero = len(arr) - np.count_nonzero(arr, axis=0)

    # adjusting/resetting ranks
    ranked_regions = ranked_regions - comp_num_zero
    ranked_regions = np.where(ranked_regions < 0, 0, ranked_regions)

    return ranked_regions

# def scale_by_sigmoid(sigmoid_ranks, top_regions_all, top_vals_all):
#     # max sigmoid value across all components for all regions
#     sigmoid_ranks_max = np.max(sigmoid_ranks, axis=1)
#     # region a max sigmoid value
#     max_sigmoid_a = np.vstack([sigmoid_ranks_max]*1000).T
#     # region b max sigmoid value
#     max_sigmoid_b = sigmoid_ranks_max[top_regions_all]
#     # average max for a and b
#     max_sigmoid_to_scale = (max_sigmoid_a + max_sigmoid_b) / 2
#     # scale similarity scores
#     top_sigmoid_scaled_vals_all = top_vals_all * max_sigmoid_to_scale
#     return top_sigmoid_scaled_vals_all

def scale_by_sigmoid(sigmoid_ranks, sim_m):
    # max sigmoid value across all components for all regions
    sigmoid_ranks_max = np.max(sigmoid_ranks, axis=1)
    # average max sigmoid value between regions a and b 
    # and scale similarity score for region pairing ab
    sim_m_sigmoid_scaled = ((sim_m * (sigmoid_ranks_max / 2)) +
                            (sim_m * (sigmoid_ranks_max / 2)).T)
    return sim_m_sigmoid_scaled

In [6]:
sigmoid_ranks_max = np.max(sigmoid_ranks, axis=1)

In [13]:
sigmoid_ranks_max[:10000]

array([0.00669285, 0.00669285, 0.00669285, ..., 0.99003643, 0.89616309,
       0.901687  ])

In [29]:
np.count_nonzero(sim_m.T==sim_m)

100000000

In [42]:
np.count_nonzero([False])

0

In [20]:
0.58178524 * (0.89616309 /2)

0.2606872291973958

In [30]:
(sim_m * (sigmoid_ranks_max[:10000] / 2)).T

array([[0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       ...,
       [0.        , 0.        , 0.        , ..., 0.49501821, 0.28799429,
        0.31777514],
       [0.        , 0.        , 0.        , ..., 0.26068723, 0.44808154,
        0.2760113 ],
       [0.        , 0.        , 0.        , ..., 0.28941734, 0.27771262,
        0.4508435 ]])

In [14]:
sim_m * (sigmoid_ranks_max[:10000] / 2) 

array([[0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       ...,
       [0.        , 0.        , 0.        , ..., 0.49501821, 0.26068723,
        0.28941734],
       [0.        , 0.        , 0.        , ..., 0.28799429, 0.44808154,
        0.27771262],
       [0.        , 0.        , 0.        , ..., 0.31777514, 0.2760113 ,
        0.4508435 ]])

In [38]:
scaled = (sim_m * (sigmoid_ranks_max[:10000] / 2)) + (sim_m * (sigmoid_ranks_max[:10000] / 2)).T

In [33]:
sim_m

array([[0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       ...,
       [0.        , 0.        , 0.        , ..., 1.        , 0.58178524,
        0.64194635],
       [0.        , 0.        , 0.        , ..., 0.58178524, 1.        ,
        0.61598454],
       [0.        , 0.        , 0.        , ..., 0.64194635, 0.61598454,
        1.        ]])

In [39]:
scaled

array([[0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       ...,
       [0.        , 0.        , 0.        , ..., 0.99003643, 0.54868152,
        0.60719248],
       [0.        , 0.        , 0.        , ..., 0.54868152, 0.89616309,
        0.55372393],
       [0.        , 0.        , 0.        , ..., 0.60719248, 0.55372393,
        0.901687  ]])

In [44]:
np.max(max_sigmoid_a[5])

0.09509070404147893

In [53]:
np.max(max_sigmoid_a.T[:, 5])

0.09509070404147893

In [ ]:
(max_sigmoid_a + max_sigmoid_a.T) / 2

In [29]:
max_sigmoid_a[:, 0][:500]

array([0.00669285, 0.00669285, 0.00669285, 0.00669285, 0.01563115,
       0.0950907 , 0.00669285, 0.00669285, 0.00669285, 0.00669285,
       0.00669285, 0.00669285, 0.00669285, 0.00669285, 0.00669285,
       0.00669285, 0.00669285, 0.00669285, 0.00669285, 0.00669285,
       0.00669285, 0.99330715, 0.00669285, 0.00669285, 0.00669285,
       0.00669285, 0.99208067, 0.75507148, 0.30132109, 0.67408158,
       0.94201285, 0.99201419, 0.97458763, 0.9893674 , 0.98798714,
       0.99141249, 0.93236805, 0.98773896, 0.95942873, 0.92728253,
       0.98378072, 0.99215018, 0.98181303, 0.98680184, 0.99044874,
       0.99100862, 0.99135837, 0.94271883, 0.5934704 , 0.98515555,
       0.87711785, 0.99303827, 0.97732353, 0.97079317, 0.98694768,
       0.97112348, 0.97188056, 0.98940703, 0.99214365, 0.93417247,
       0.98702578, 0.9886737 , 0.94980634, 0.66449618, 0.86614847,
       0.85685584, 0.98452506, 0.61085736, 0.98840783, 0.98016183,
       0.99198088, 0.99294394, 0.987781  , 0.99226147, 0.98991

In [3]:
val_path = '/home/ctrader/region_similarity/data/rank_based/regions_25kb_CT20221206.DHSsignalzscores.hdf5'

In [4]:
# load the data
print('Loading data...', flush=True)
val_hdf5 = h5py.File(val_path, 'r')
val_arr = val_hdf5['base_arr'][:]
val_hdf5.close()

# for epilogos scores, setting negative values to 0
val_arr = np.where(val_arr < 0, 0, val_arr)

print('Ranking...', flush=True)
# rank regions
ranked_regions = rank_regions(val_arr)

# max rankings
max_ranks = np.max(ranked_regions, axis=0)

print('Sigmoid...', flush=True)
# sigmoid of rank for all regions
sigmoid_ranks = sigmoid_func(ranked_regions)

print('Euclidean...', flush=True)
# euclidean distance matrix
distance_m = squareform(pdist(sigmoid_ranks[:10000], metric='euclidean'))

# find the max distance
max_distance = np.max(distance_m)

print('Similarity...', flush=True)
# sim matrix
# scaling between 0 and 1 by the max observed value
sim_m = 1 - (distance_m / max_distance)
del distance_m

# regions with 0 datapoints
total_val_per_region = np.sum(val_arr[:10000], axis=1)
regions_0_val = np.where(total_val_per_region == 0)[0]

# setting similarity values in regions with no datapoints to 0
sim_m[regions_0_val] = 0
sim_m[:, regions_0_val] = 0

Loading data...
Ranking...
Sigmoid...
Euclidean...
Similarity...


In [14]:
sim_m[0]

array([0., 0., 0., ..., 0., 0., 0.])

**DHS signal**

In [2]:
mapped_counts_filename='/home/ctrader/region_similarity/data/rank_based/regions_25kb_CT20221205.DHSsignalmapped.bed'


In [3]:
    # load count data
    data = []
    f = open(mapped_counts_filename, 'r')
    for line in f.readlines()[1:]:  # not header
        data.append(re.split('[\t|]', line.strip()))
    f.close()
    data = np.array(data)

    # seperating counts, means, and scores
    counts = data[:, 3::3].astype(int)
    means = data[:, 4::3]
    means = np.where(means=='NAN', np.nan, means).astype('float32')

    # find mean and std for each factor
    factor_means = np.array([])
    factor_std = np.array([])
    for factor in range(len(data[0, 5::3])):
        factor_data = data[:, 5 + (factor * 3)][data[:, 5 + (factor * 3)] != '']
        factor_scores = np.array(sum(list(np.char.rsplit(factor_data, ';')), [])).astype('float32')
        factor_means = np.append(factor_means, np.mean(factor_scores))
        factor_std = np.append(factor_std, np.std(factor_scores))

    # z score
    z = ((means - factor_means) / factor_std * np.sqrt(counts))
    # one sided
    z = np.where(((z < 0) | (np.isnan(z))), 0, z)

**STRING interconnectiveness**

In [14]:
filename = '/home/ctrader/region_similarity/data/9606.protein.links.full.v11.5.txt'
f = open(filename, 'r')
header = f.readline().strip().split(' ')
f.close()
string_genes_arr = np.loadtxt(filename, dtype=str, skiprows=1, usecols=[0, 1])
string_vals_arr = np.loadtxt(filename, dtype=int, skiprows=1, usecols=np.arange(2, len(header)))

In [19]:
val_header = header[2:]

In [32]:
np.sort(val_header)

array(['coexpression', 'coexpression_transferred', 'combined_score',
       'cooccurence', 'database', 'database_transferred', 'experiments',
       'experiments_transferred', 'fusion', 'homology', 'neighborhood',
       'neighborhood_transferred', 'textmining', 'textmining_transferred'],
      dtype='<U24')

In [39]:
neigh = [0, 1]
fusion = [2]
cooccur = [3]
homology = [4]
coex = [5, 6]
exper = [7, 8]
database = [9, 10]
textmin = [11, 12]
combined_score = [13]
metric_inds = [coex, cooccur, database, exper, fusion, homology, neigh, textmin, combined_score]

In [38]:
unique_string_genes = np.unique(string_genes_arr)

In [41]:
#indicies for each gene in alphabetical order
gene2ind = {}
for gene, ind in zip(unique_string_genes, np.arange(len(unique_string_genes))):
    gene2ind[gene] = ind

In [64]:
#gene ids -> indices
string_gene_inds_arr = np.array(list(map(gene2ind.get, string_genes_arr.flatten())), 
                                dtype=int).reshape(string_genes_arr.shape)

In [157]:
#for each metric, populate [metric, protein1, protein2] array
metric_protein1_protein2 = []
for col_inds in metric_inds:
    print(np.array(val_header)[col_inds])
    #combining human and transferred scores if applicable
    val_arr = np.max(string_vals_arr[:, col_inds], axis=1)
    metric_sparse = sp.sparse.coo_matrix((val_arr, 
                                          (string_gene_inds_arr[:, 0], string_gene_inds_arr[:, 1])))
    metric_protein1_protein2.append(metric_sparse.toarray())
metric_protein1_protein2 = np.array(metric_protein1_protein2)

['coexpression' 'coexpression_transferred']
['cooccurence']
['database' 'database_transferred']
['experiments' 'experiments_transferred']
['fusion']
['homology']
['neighborhood' 'neighborhood_transferred']
['textmining' 'textmining_transferred']
['combined_score']


**Improving STRING similarity calculation**

In [318]:
#making a dictionary to map genes to regions. Each gene will map to an arrray of regions
def map_genes_to_regions(all_regions_gene_ids):
    gene2region = {}
    for genes, region in zip(all_regions_gene_ids, range(len(all_regions_gene_ids))):
        if len(genes) > 1:
            for gene in genes:
                if gene not in gene2region:
                    gene2region[gene] = np.array([region])
                else:
                    gene2region[gene] = np.append(gene2region[gene], [region])
    return gene2region

#making dictionaries to map a gene to similar genes and corresponding scores
def map_genes_to_genes(metric_top1000):
    gene2genes = {}
    gene2scores = {}
    for row in metric_top1000:
        gene0 = row[0]
        gene1 = row[1]
        score = int(row[2])
        if gene0 not in gene2genes:
            gene2genes[gene0] = np.array([gene1])
            gene2scores[gene0] = np.array([score])
        else:
            gene2genes[gene0] = np.append(gene2genes[gene0], [gene1])
            gene2scores[gene0] = np.append(gene2scores[gene0], [score])
    return gene2genes, gene2scores

#converting string scores to a num_regions x num_saved_recommended_regions matrix
def create_sim_matrix(all_regions_gene_ids, metric_top1000, gene2region, gene2genes, gene2scores, starting_region):
    genes = np.unique(all_regions_gene_ids[starting_region]) ####### unique
    #if there are no genes
    if len(genes) == 0:
        region_scores_to_m = np.zeros(1000, dtype=int)
        similar_regions_to_m = np.arange(1000)
        origin_gene_tracker_to_m = np.array(['nan']*1000)
        destination_gene_tracker_to_m = np.array(['nan']*1000)
    #if there is at least 1 gene
    else:
        region_scores_init = np.array([], dtype=int)
        similar_genes_init = np.array([])
        origin_gene_tracker_init = np.array([])
        for gene in genes:
            #finding similar genes/scores based on origin gene
            region_scores_init = np.append(region_scores_init, gene2scores[gene])
            similar_genes_init = np.append(similar_genes_init, gene2genes[gene])
            origin_gene_tracker_init = np.append(origin_gene_tracker_init, np.array([gene]*len(gene2scores[gene])))

        region_scores = np.array([], dtype=int)
        similar_regions = np.array([], dtype=int)
        origin_gene_tracker = np.array([])
        destination_gene_tracker = np.array([])
        for gene, score, origin_gene in zip(similar_genes_init, region_scores_init, origin_gene_tracker_init):
            #if gene in some region, add scores/regions to final arrays
            if gene in gene2region:
                gene_regions = np.unique(gene2region[gene]) ######### unique
                similar_regions = np.append(similar_regions, gene_regions)
                region_scores = np.append(region_scores, np.array([score]*len(gene_regions)))
                origin_gene_tracker = np.append(origin_gene_tracker, np.array([origin_gene]*len(gene_regions)))
                destination_gene_tracker = np.append(destination_gene_tracker, np.array([gene]*len(gene_regions)))
        
        #sort scores
        sort_order = np.argsort(-region_scores)
        region_scores = region_scores[sort_order]
        similar_regions = similar_regions[sort_order]
        origin_gene_tracker = origin_gene_tracker[sort_order]
        destination_gene_tracker = destination_gene_tracker[sort_order]

        #finding first occurrence
        _, first_occurrence_ind = np.unique(similar_regions, return_index=True)
        first_occurrence_ind_sorted = np.sort(first_occurrence_ind)

        #values with first occurrence
        similar_regions_to_m = similar_regions[first_occurrence_ind_sorted]
        region_scores_to_m = region_scores[first_occurrence_ind_sorted]
        origin_gene_tracker_to_m = origin_gene_tracker[first_occurrence_ind_sorted]
        destination_gene_tracker_to_m = destination_gene_tracker[first_occurrence_ind_sorted]

        #removing starting region values, take top 1000 regions/scores
        starting_region_mask = similar_regions_to_m!=starting_region
        similar_regions_to_m = similar_regions_to_m[starting_region_mask][:1000]
        region_scores_to_m = region_scores_to_m[starting_region_mask][:1000]
        origin_gene_tracker_to_m = origin_gene_tracker_to_m[starting_region_mask][:1000]
        destination_gene_tracker_to_m = destination_gene_tracker_to_m[starting_region_mask][:1000]

        arr_len = len(region_scores_to_m)
        if arr_len < 1000:
            region_scores_to_m = np.append(region_scores_to_m, np.zeros(1000-arr_len, dtype=int))
            similar_regions_to_m = np.append(similar_regions_to_m, np.array([starting_region]*(1000-arr_len)))
            origin_gene_tracker_to_m = np.append(origin_gene_tracker_to_m, np.array(['nan']*(1000-arr_len)))
            destination_gene_tracker_to_m = np.append(destination_gene_tracker_to_m, np.array(['nan']*(1000-arr_len)))
            
    return np.append(np.append(np.append(region_scores_to_m.astype(str), similar_regions_to_m.astype(str)), origin_gene_tracker_to_m), destination_gene_tracker_to_m)

#ranking and scaling data
def scale_data(data, max_poss):
    data_to_rank = data[data!=0]
    max_obs = np.max(data_to_rank)
    rank_data = rankdata(data_to_rank, method='average')
    rank_data = rank_data/(len(data_to_rank))
    rank_data = rank_data * max_obs/max_poss

    original_data_shape = data.shape
    data = data.astype(float).flatten()
    np.put(data, np.where(data!=0)[0], rank_data)
    data = data.reshape(original_data_shape)
    return data

In [75]:
len(np.unique(all_regions_gene_ids[40]))

4

In [6]:
metric_path = '/home/ctrader/region_similarity/data/STRING/9606.protein.links.v11.5.textmining_human_transferred_max.all_associations.CT20220711.txt'
gene_id_mapped_path = '/home/ctrader/region_similarity/data/count_based/regions_25kb_CT20221027.proteinidmapped.bed'

In [7]:
print('Loading data...', flush=True)
#load metric data
f = open(metric_path, 'r')
metric_top1000 = []
for line in f.readlines():
    line = line.strip().split('\t')
    metric_top1000.append(line)
metric_top1000 = np.array(metric_top1000)
f.close()

Loading data...


In [10]:
#find the mapped gene ids that have metric associations listed
unique_gene_codes_with_associations = set(metric_top1000[:, 0])

In [11]:
f = open(gene_id_mapped_path, 'r')
all_regions_gene_ids = []
for line in f.readlines():
    region_gene_ids = []
    geneids = re.split('\t|;|\|', line.strip())[3:]
    for geneid in geneids:
        geneid = geneid.split('.')[0] #removing everything after .
        if geneid in unique_gene_codes_with_associations:
            region_gene_ids.append(geneid)
    all_regions_gene_ids.append(region_gene_ids)
f.close()

In [13]:
print('Mapping genes to regions, genes, and scores...', flush=True)
#making a dictionary to map genes to regions. Each gene will map to an arrray of regions
gene2region = map_genes_to_regions(all_regions_gene_ids)

Mapping genes to regions, genes, and scores...


In [14]:
#making dictionaries to map a gene to similar genes and corresponding scores
gene2genes, gene2scores = map_genes_to_genes(metric_top1000)

In [16]:
metric_top1000

array([['ENSP00000000233', 'ENSP00000379496', '103'],
       ['ENSP00000000233', 'ENSP00000314067', '61'],
       ['ENSP00000000233', 'ENSP00000263116', '101'],
       ...,
       ['ENSP00000485678', 'ENSP00000335660', '0'],
       ['ENSP00000485678', 'ENSP00000300127', '0'],
       ['ENSP00000485678', 'ENSP00000310488', '0']], dtype='<U15')

In [319]:
print('Finding similar regions...', flush=True)
func = partial(create_sim_matrix, all_regions_gene_ids, metric_top1000, gene2region, gene2genes, gene2scores)
with closing(Pool(1)) as pool:
    top_1000_similar_inds = pool.map(func, [0, 70725, 70726])
pool.join()
top_1000_similar_inds = np.array(top_1000_similar_inds)

Finding similar regions...


In [322]:
func = partial(create_sim_matrix, all_regions_gene_ids, metric_top1000, gene2region, gene2genes, gene2scores)

t = time.time()
for i in np.arange(91000, 92000):
    top_1000_similar_inds = func(i)
print(time.time()-t)

94.24832725524902


In [323]:
94.24832725524902 * 100 / 10 / 60 

15.708054542541504

In [181]:
100000 / 100 / 60

16.666666666666668

In [134]:
np.unique(all_regions_gene_ids[40])

array(['ENSP00000329137', 'ENSP00000353094', 'ENSP00000368496',
       'ENSP00000368538'], dtype='<U15')

In [295]:
t = time.time()

starting_region = 70725
genes = np.unique(all_regions_gene_ids[starting_region])
#if there are no genes
if len(genes) == 0:
    region_scores_to_m = np.zeros(1000, dtype=int)
    similar_regions_to_m = np.arange(1000)
    origin_gene_tracker_to_m = np.array(['nan']*1000)
    destination_gene_tracker_to_m = np.array(['nan']*1000)
#if there is at least 1 gene
else:
    print('step 1', time.time()-t)
    region_scores_init = np.array([], dtype=int)
    similar_genes_init = np.array([])
    origin_gene_tracker_init = np.array([])
    for gene in genes:
        #finding similar genes/scores based on origin gene
        region_scores_init = np.append(region_scores_init, gene2scores[gene])
        similar_genes_init = np.append(similar_genes_init, gene2genes[gene])
        origin_gene_tracker_init = np.append(origin_gene_tracker_init, np.array([gene]*len(gene2scores[gene])))

    print('step 2', time.time()-t)

    region_scores = np.array([], dtype=int)
    similar_regions = np.array([], dtype=int)
    origin_gene_tracker = np.array([])
    destination_gene_tracker = np.array([])
    for gene, score, origin_gene in zip(similar_genes_init, region_scores_init, origin_gene_tracker_init):
        #if gene in some region, add scores/regions to final arrays
        if gene in gene2region:
            gene_regions = np.unique(gene2region[gene])
            similar_regions = np.append(similar_regions, gene_regions)
            region_scores = np.append(region_scores, np.array([score]*len(gene_regions)))
            origin_gene_tracker = np.append(origin_gene_tracker, np.array([origin_gene]*len(gene_regions)))
            destination_gene_tracker = np.append(destination_gene_tracker, np.array([gene]*len(gene_regions)))

    print('step 3', time.time()-t)

    #sort scores
    sort_order = np.argsort(-region_scores)
    region_scores = region_scores[sort_order]
    similar_regions = similar_regions[sort_order]
    origin_gene_tracker = origin_gene_tracker[sort_order]
    destination_gene_tracker = destination_gene_tracker[sort_order]

    print('step 4', time.time()-t)

#     #find first occurrence/score of each region
#     region_scores_to_m_old = np.array([], dtype=int)
#     similar_regions_to_m_old = np.array([], dtype=int)
#     origin_gene_tracker_to_m_old = np.array([])
#     destination_gene_tracker_to_m_old = np.array([])
#     for region, score, original_gene, destination_gene in zip(similar_regions, region_scores, origin_gene_tracker, destination_gene_tracker):
#         if ((region not in similar_regions_to_m_old) & (region != starting_region)):
#             similar_regions_to_m_old = np.append(similar_regions_to_m_old, [region])
#             region_scores_to_m_old = np.append(region_scores_to_m_old, [score])
#             origin_gene_tracker_to_m_old = np.append(origin_gene_tracker_to_m_old, [original_gene])
#             destination_gene_tracker_to_m_old = np.append(destination_gene_tracker_to_m_old, [destination_gene])

    #finding first occurrence
    _, first_occurrence_ind = np.unique(similar_regions, return_index=True)
    first_occurrence_ind_sorted = np.sort(first_occurrence_ind)

    #values with first occurrence
    similar_regions_to_m = similar_regions[first_occurrence_ind_sorted]
    region_scores_to_m = region_scores[first_occurrence_ind_sorted]
    origin_gene_tracker_to_m = origin_gene_tracker[first_occurrence_ind_sorted]
    destination_gene_tracker_to_m = destination_gene_tracker[first_occurrence_ind_sorted]

    #removing starting region values
    starting_region_mask = similar_regions_to_m!=starting_region
    similar_regions_to_m = similar_regions_to_m[starting_region_mask]
    region_scores_to_m = region_scores_to_m[starting_region_mask]
    origin_gene_tracker_to_m = origin_gene_tracker_to_m[starting_region_mask]
    destination_gene_tracker_to_m = destination_gene_tracker_to_m[starting_region_mask]

    print('step 5', time.time()-t)


step 1 0.0008213520050048828
step 2 0.00697779655456543
step 3 2.9231913089752197
step 4 2.926593065261841
step 5 2.930111885070801


In [294]:
similar_regions_to_m_old.shape

(10038,)

In [296]:
similar_regions_to_m.shape

(10038,)

In [268]:
t = time.time()
#finding first occurrence
_, first_occurrence_ind = np.unique(similar_regions, return_index=True)
first_occurrence_ind_sorted = np.sort(first_occurrence_ind)

#values with first occurrence
similar_regions_to_m_new = similar_regions[first_occurrence_ind_sorted]
region_scores_to_m_new = region_scores[first_occurrence_ind_sorted]
origin_gene_tracker_to_m_new = origin_gene_tracker[first_occurrence_ind_sorted]
destination_gene_tracker_to_m_new = destination_gene_tracker[first_occurrence_ind_sorted]

#removing starting region values
starting_region_mask = similar_regions_to_m_new!=starting_region
similar_regions_to_m_new = similar_regions_to_m_new[starting_region_mask]
region_scores_to_m_new = region_scores_to_m_new[starting_region_mask]
origin_gene_tracker_to_m_new = origin_gene_tracker_to_m_new[starting_region_mask]
destination_gene_tracker_to_m_new = destination_gene_tracker_to_m_new[starting_region_mask]

print(time.time()-t)

0.0019702911376953125


In [297]:
similar_regions_to_m_old.shape

(10038,)

In [290]:
similar_regions_to_m.shape

(10038,)

In [298]:
for i, j in zip(origin_gene_tracker_to_m_old, origin_gene_tracker_to_m):
    if i != j:
        print(i, j)

In [299]:
for i, j in zip(destination_gene_tracker_to_m_old, destination_gene_tracker_to_m):
    if i != j:
        print(i, j)

In [313]:
np.sum(region_scores_to_m_old - region_scores_to_m)

0

In [312]:
np.sum(similar_regions_to_m - similar_regions_to_m_old)

0

In [211]:
destination_gene_tracker_to_m.shape

(5619,)

In [193]:
np.unique(similar_regions)[:10]

array([31, 32, 33, 34, 35, 38, 39, 40, 41, 42])

In [184]:
vals, inds = np.unique(np.array([1, 2, 4, 1, 43, 2, 5]), return_index=True)

In [186]:
np.sort(inds)

array([0, 1, 2, 4, 6])

In [168]:
gene2region[gene]

KeyError: 'ENSP00000356946'

In [167]:
sort_order.shape

(6144,)

In [164]:
region_scores_to_m.shape

(5619,)

In [162]:
region_scores_to_m.shape

(5619,)

In [124]:
similar_genes_init.shape[0] * 0.008234739303588867

14.311976909637451

In [150]:
len(similar_genes_init)

1738

In [126]:
np.unique(similar_genes_init).shape

(1646,)

In [137]:
list(map(gene2region.get, ['asasasa']))

[None]

In [151]:
len(region_scores)

126221

In [142]:
t = list(map(gene2region.get, similar_genes_init))

In [143]:
t

[array([82323, 82323, 82323, 82323, 82323, 82323, 82323, 82323, 82323,
        82323, 82323, 82323, 82323, 82323, 82323, 82323, 82323, 82324,
        82324, 82324, 82324, 82324, 82324, 82324, 82324, 82324, 82324,
        82324, 82324, 82324, 82324, 82324, 82324, 82324, 82325, 82325,
        82325, 82325, 82325, 82325, 82325, 82325, 82325, 82325, 82325,
        82325, 82325, 82325]),
 array([18015, 18015, 18015, 18015, 18015, 18015, 18015, 18015, 18015]),
 array([21544, 21544, 21544, 21544, 21544, 21544, 21544, 21544, 21544,
        21544, 21544, 21545, 21545, 21545, 21545, 21545, 21546, 21546,
        21546, 21546, 21546]),
 array([866, 866, 866, 866, 866, 866, 866, 866, 866, 866, 866, 866, 866,
        866, 866, 866, 866]),
 array([61552, 61552, 61552, 61553, 61553, 61553, 61553, 61553, 61553,
        61553, 61553, 61553, 61553, 61553, 61553, 61553, 61553, 61553,
        61553, 61553, 61554, 61554, 61554, 61554, 61554, 61554, 61554,
        61554, 61554, 61554, 61554, 61554, 61554, 61

In [170]:
region_scores = np.array([], dtype=int)
similar_regions = np.array([], dtype=int)
origin_gene_tracker = np.array([])
destination_gene_tracker = np.array([])
for gene, score, origin_gene in zip(similar_genes_init, region_scores_init, origin_gene_tracker_init):
    #if gene in some region, add scores/regions to final arrays
    if gene in gene2region:
        t = time.time()
        gene_regions = np.unique(gene2region[gene]) #####unique
        
        similar_regions = np.append(similar_regions, gene_regions)
        region_scores = np.append(region_scores, np.array([score]*len(gene_regions)))
        origin_gene_tracker = np.append(origin_gene_tracker, np.array([origin_gene]*len(gene_regions)))
        destination_gene_tracker = np.append(destination_gene_tracker, np.array([gene]*len(gene_regions)))

In [147]:
similar_regions[:48]

array([82323, 82323, 82323, 82323, 82323, 82323, 82323, 82323, 82323,
       82323, 82323, 82323, 82323, 82323, 82323, 82323, 82323, 82324,
       82324, 82324, 82324, 82324, 82324, 82324, 82324, 82324, 82324,
       82324, 82324, 82324, 82324, 82324, 82324, 82324, 82325, 82325,
       82325, 82325, 82325, 82325, 82325, 82325, 82325, 82325, 82325,
       82325, 82325, 82325])

In [148]:
region_scores[:48]

array([233, 233, 233, 233, 233, 233, 233, 233, 233, 233, 233, 233, 233,
       233, 233, 233, 233, 233, 233, 233, 233, 233, 233, 233, 233, 233,
       233, 233, 233, 233, 233, 233, 233, 233, 233, 233, 233, 233, 233,
       233, 233, 233, 233, 233, 233, 233, 233, 233])

In [117]:
origin_gene_tracker = np.append(origin_gene_tracker, np.array([origin_gene]*1000))

In [118]:
t = time.time()
similar_regions = np.append(similar_regions, gene_regions)
print(time.time() - t)
region_scores = np.append(region_scores, np.array([score]*1000))
print(time.time() - t)

0.0009291172027587891
0.0014922618865966797


In [97]:
origin_gene_tracker = np.append(origin_gene_tracker, np.array([origin_gene]*len(gene_regions)))

In [39]:
all_regions_gene_ids[55655]

['ENSP00000332371',
 'ENSP00000332371',
 'ENSP00000332371',
 'ENSP00000332371',
 'ENSP00000332371',
 'ENSP00000332371',
 'ENSP00000332371',
 'ENSP00000332371',
 'ENSP00000332371',
 'ENSP00000332371',
 'ENSP00000332371',
 'ENSP00000332371',
 'ENSP00000332371',
 'ENSP00000332371',
 'ENSP00000332371',
 'ENSP00000332371',
 'ENSP00000332371',
 'ENSP00000332371',
 'ENSP00000332371',
 'ENSP00000332371',
 'ENSP00000332371',
 'ENSP00000332371',
 'ENSP00000332371',
 'ENSP00000332371',
 'ENSP00000332371',
 'ENSP00000332371',
 'ENSP00000332371',
 'ENSP00000332371',
 'ENSP00000332371',
 'ENSP00000332371',
 'ENSP00000332371',
 'ENSP00000332371',
 'ENSP00000332371',
 'ENSP00000332371',
 'ENSP00000332371',
 'ENSP00000332371',
 'ENSP00000332371',
 'ENSP00000332371',
 'ENSP00000332371',
 'ENSP00000332371',
 'ENSP00000332371',
 'ENSP00000332371',
 'ENSP00000332371',
 'ENSP00000332371',
 'ENSP00000332371',
 'ENSP00000332371',
 'ENSP00000332371',
 'ENSP00000332371',
 'ENSP00000332371',
 'ENSP00000332371',


In [282]:
m = []
for i in all_regions_gene_ids:
    m.append(len(np.unique(i)))
m = np.array(m)

In [284]:
m[70725]

22

In [72]:
np.unique(all_regions_gene_ids[40]).shape

(4,)

In [69]:
m[40]

66

In [58]:
m[40]

66